# Limpeza e Preparação dos Dados 🏗️ 🎲

<pre>Algumas organizações oferecem bibliotecas para facilitar a criação de consultas, integrando-as ao nosso código.</pre>

## Explorando as estatísticas de gênero do Banco Mundial

<pre>O Banco Mundial oferece biblioteca em Python para consultar seus bancos de dados.</pre>

👉 mais detalhes, consulte o pacote <a href='https://pypi.org/project/wbgapi/'>wbgapi</a> e sua <a href='https://blogs.worldbank.org/opendata/introducing-wbgapi-new-python-package-accessing-world-bank-data'>documentação</a>.

<pre>Vamos instalar e importar as bibliotecas que serão utilizadas neste caderno.</pre>

In [1]:
!pip install wbgapi --quiet
!pip install pandas --quiet

In [2]:
import wbgapi as wb
import pandas as pd

<pre>Execute as 3 células abaixo para montar os DataFrames <b>labor_force_country</b> e <b>labor_force_income</b>.</pre>

In [3]:
# carregando dados de taxa de participação da força laboral feminina
female_labor_force = wb.data.DataFrame('SL.TLF.ACTI.FE.ZS', time = 2019).squeeze()

# carregando dados de taxa de participação da força laboral masculina
male_labor_force = wb.data.DataFrame('SL.TLF.ACTI.MA.ZS', time = 2019).squeeze()

In [4]:
# carregando dados econômicos
economy_info = wb.economy.info()

# carregando dados de países
country_codes = [pais.get('id') 
                 for pais in economy_info.items
                 if pais.get('region') != ''] # códigos que não são países têm o campo region e income em branco

# filtrando apenas os dados referentes a países nos dados de forças laborais
female_labor_force_country = female_labor_force[country_codes]
male_labor_force_country = male_labor_force[country_codes]

# criando um dataframe com duas colunas, uma de força laboral feminina e outra de força laboral masculina
labor_force_country = pd.DataFrame({"female_rate": female_labor_force_country,
                                    "male_rate": male_labor_force_country})

# criando uma coluna que armazena a diferença entre a força laboral masculina e a força laboral feminina
labor_force_country['rate_gap'] = labor_force_country["male_rate"] - labor_force_country["female_rate"]

In [5]:
# carregando dados de renda
income_info = wb.income.info()

# carregando dados de faixa de renda
income_levels = [item.get('id') for item in income_info.items]

# filtrando apenas os dados referentes a faixa de renda nos dados de forças laborais
female_labor_force_income = female_labor_force[income_levels]
male_labor_force_income = male_labor_force[income_levels]

# criando um dataframe com duas colunas, uma de força laboral feminina e outra de força laboral masculina
labor_force_income = pd.DataFrame({"female_rate": female_labor_force_income,
                                   "male_rate": male_labor_force_income})

# criando uma coluna que armazena a diferença entre a força laboral masculina e a força laboral feminina
labor_force_income['rate_gap'] = labor_force_income["male_rate"] - labor_force_income["female_rate"]

<pre>Vamos começar removendo, dos DataFrames <b>labor_force_country</b> e <b>labor_force_income</b>, as linhas sem dados disponíveis (no caso, com valores <b>NaN</b>).</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country = labor_force_country.dropna()
labor_force_income = labor_force_income.dropna()
```

<br/>

</details>

<pre>Para facilitar a compreensão das tabelas, vamos acrescentar colunas aos DataFrames, com os nomes dos países ou dos grupos de renda, de acordo com o escopo de cada DataFrame.</pre>

<pre>Primeiro, crie dois dicionários que associam:
    1 - o código do país ao seu nome em um, atribuindo o resultado à variável <b>country_codes_names</b>; e 
    2 - o código do grupo de renda ao seu nome em outro, atribuindo o resultado à variável <b>income_level_names</b>.</pre>

👉 dica: use os objetos `economy_info` e `income_info`.

<details>
  <summary>Resposta</summary>

<br/>

```python
country_codes_names = {pais.get('id'): pais.get('value') 
                       for pais in economy_info.items 
                       if pais.get('region') != ''}

income_level_names = {grupo.get('id'): grupo.get('value')
                      for grupo in income_info.items}
```

<br/>

</details>

<pre>É assim que deve parecer o dicionário armazenado na variável <b>country_codes_names</b>:</pre>

```python
{'ABW': 'Aruba',
 'AFG': 'Afghanistan',
 'AGO': 'Angola',
 'ALB': 'Albania',
 'AND': 'Andorra',
 'ARE': 'United Arab Emirates', ...}
```

<pre>Similarmente, assim que deve parecer o dicionário armazenado na variável <b>income_level_names</b>:</pre>

```python
{'HIC': 'High income',
 'INX': 'Not classified',
 'LIC': 'Low income',
 'LMC': 'Lower middle income',
 'LMY': 'Low & middle income',
 'MIC': 'Middle income',
 'UMC': 'Upper middle income'}
```

<pre>Agora, usando o método <b>map</b>, crie as novas colunas nos DataFrames, da seguinte forma:
    1 - coluna <b>country</b>, no DataFrame <b>labor_force_country</b>, com o nome por extenso dos países correspondentes; e
    2 - coluna <b>group</b>, no DataFrame <b>labor_force_income</b>, com o nome por extenso dos grupos de renda correspondentes.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country['country'] = labor_force_country.index.map(country_codes_names)
labor_force_income['group'] = labor_force_income.index.map(income_level_names)
```

<br/>

</details>

<pre>Agora, obtenha a média para o grupo formado por <b>Brasil</b>, <b>Argentina</b>, <b>Uruguai</b> e <b>Paraguai</b>, bem como outra média para o grupo formado por <b>Estados Unidos</b> e <b>Canadá</b>, de todas as colunas do DataFrame <b>labor_force_country</b>.</pre>

👉 dica: para evitar que o pandas realize uma média aritmética em um atributo não numérico, use o parâmetro `numeric_only` do método `mean`.

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.loc[["BRA", "ARG", "URY", "PRY"]].mean(numeric_only=True)
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.loc[["CAN", "USA"]].mean(numeric_only=True)
```

<br/>

</details>

<pre>Por fim, identifique os países com diferença nas taxas de participação maiores que 50 pontos percentuais, mostrando o resultado por ordem decrescente na taxa de participação.</pre>

👉 dica: use o DataFrame `labor_force_country`, filtre usando indexação booleana na coluna `rate_gap`, e selecione as colunas `country` e `rate_gap` para aparecerem no resultado.

<details>
  <summary>Resposta</summary>

<br/>

```python
resultado = labor_force_country.loc[labor_force_country['rate_gap'] > 50, ['country', 'rate_gap']]
resultado.sort_values('rate_gap', ascending=False)
```

<br/>

</details>

<pre>Agora, liste os países que apresentam as 10 maiores taxas de participação masculina na força trabalho e seus respectivos valores, ordenando-os de forma descrescente.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
labor_force_country.sort_values("male_rate", ascending = False).head(10)[['country', 'male_rate']]
```

<br/>

</details>